In [28]:
import pandas as pd
import numpy as np

In [29]:

path= r'C:\Users\nadia\OneDrive\Desktop\Leticia\Documentos\Projetos\cliente_confeitaria\Analise_financeira_confeitaria\Dados'
ifood = pd.read_csv(path +r'\originais\relatorio_repasses_ifood.csv', sep=',', encoding='utf-8',dtype=str)

In [30]:
print(ifood.shape)
print(ifood.columns.tolist())
display(ifood.head(10))
display(ifood.tail(5))

for col in ifood.columns:
    print(col)
    print(ifood[col].str.replace(r"\d", "9", regex=True).value_counts().head(5), "\n")

(74, 7)
['data_repasse', 'periodo_vendas', 'qtd_pedidos', 'valor_vendas', 'comissao_ifood', 'taxa_pagamento', 'valor_repasse']


,data_repasse,periodo_vendas,qtd_pedidos,valor_vendas,comissao_ifood,taxa_pagamento,valor_repasse
0,2025-06-04,21/05-27/05,16,949.08,182.76,37.43,728.89
1,2025-06-11,28/05-03/06,17,960.41,184.93,37.88,737.60
2,2025-06-18,04/06-10/06,15,860.91,165.78,33.95,661.18
3,2025-06-25,11/06-17/06,14,824.84,158.83,32.53,633.48
4,2025-07-02,18/06-24/06,16,913.40,175.89,36.02,701.49
5,2025-07-09,25/06-01/07,13,730.83,140.73,28.82,561.28
6,2025-07-16,02/07-08/07,12,707.07,136.15,27.89,543.03
7,2025-07-23,09/07-15/07,14,811.18,156.20,31.99,622.99
8,2025-07-30,16/07-22/07,13,743.52,143.17,29.33,571.02
9,2025-08-06,23/07-29/07,13,780.74,150.34,30.79,599.61


,data_repasse,periodo_vendas,qtd_pedidos,valor_vendas,comissao_ifood,taxa_pagamento,valor_repasse
69,2026-09-30,16/09-22/09,11,655.03,126.14,25.83,503.06
70,2026-10-07,23/09-29/09,15,878.54,169.17,34.65,674.72
71,2026-10-14,30/09-06/10,14,804.45,154.90,31.73,617.82
72,2026-10-21,07/10-13/10,13,743.34,143.14,29.32,570.88
73,2026-10-28,14/10-20/10,13,755.26,145.43,29.79,580.04


data_repasse
data_repasse
9999-99-99    74
Name: count, dtype: int64 

periodo_vendas
periodo_vendas
99/99-99/99    74
Name: count, dtype: int64 

qtd_pedidos
qtd_pedidos
99    73
9      1
Name: count, dtype: int64 

valor_vendas
valor_vendas
999.99     63
9999.99    11
Name: count, dtype: int64 

comissao_ifood
comissao_ifood
999.99    73
99.99      1
Name: count, dtype: int64 

taxa_pagamento
taxa_pagamento
99.99    74
Name: count, dtype: int64 

valor_repasse
valor_repasse
999.99     70
9999.99     4
Name: count, dtype: int64 



In [31]:
# ---------------------------------------------------------------
# 1. DATA DO REPASSE (aaaa-mm-dd)
# ---------------------------------------------------------------
ifood["data_repasse"] = pd.to_datetime(ifood["data_repasse"], format="%Y-%m-%d", errors="coerce")

# ---------------------------------------------------------------
# 2. PERÍODO -> duas colunas de data, com o ano inferido
# ---------------------------------------------------------------
partes = ifood["periodo_vendas"].str.split("-", expand=True)    # "21/05" | "27/05"
ini_dm, fim_dm = partes[0].str.strip(), partes[1].str.strip()

ano_rep = ifood["data_repasse"].dt.year
mes_rep = ifood["data_repasse"].dt.month
mes_fim = fim_dm.str[-2:].astype(int)
mes_ini = ini_dm.str[-2:].astype(int)

# fim do período: mesmo ano do repasse, a não ser que o mês do fim seja "depois" do mês do repasse
ano_fim = np.where(mes_fim > mes_rep, ano_rep - 1, ano_rep)
# início do período: mesmo ano do fim, a não ser que o mês do início seja "depois" do mês do fim
ano_ini = np.where(mes_ini > mes_fim, ano_fim - 1, ano_fim)

ifood["periodo_inicio"] = pd.to_datetime(ini_dm + "/" + pd.Series(ano_ini, index=ifood.index).astype(str),
                                         format="%d/%m/%Y", errors="coerce")
ifood["periodo_fim"] = pd.to_datetime(fim_dm + "/" + pd.Series(ano_fim, index=ifood.index).astype(str),
                                      format="%d/%m/%Y", errors="coerce")
ifood = ifood.drop(columns="periodo_vendas")

# ---------------------------------------------------------------
# 3. VALORES (ponto decimal -> número direto)
# ---------------------------------------------------------------
for col in ["qtd_pedidos", "valor_vendas", "comissao_ifood", "taxa_pagamento", "valor_repasse"]:
    ifood[col] = pd.to_numeric(ifood[col], errors="coerce")

print("Falhas (vazios depois da conversão):")
print(ifood.isna().sum())

# ---------------------------------------------------------------
# 4. CONFERÊNCIAS
# ---------------------------------------------------------------
# a) sua hipótese: vendas - comissão - taxa = repasse
ifood["dif_conta"] = (ifood["valor_vendas"] - ifood["comissao_ifood"]
                      - ifood["taxa_pagamento"] - ifood["valor_repasse"]).round(2)
print("\nLinhas em que a conta não fecha:", (ifood["dif_conta"].abs() > 0.01).sum())

# b) cada período tem 7 dias?
ifood["dias_periodo"] = (ifood["periodo_fim"] - ifood["periodo_inicio"]).dt.days + 1
print("\nDias por período:")
print(ifood["dias_periodo"].value_counts())

# c) quantos dias entre o fim do período e o repasse?
ifood["dias_ate_repasse"] = (ifood["data_repasse"] - ifood["periodo_fim"]).dt.days
print("\nDias entre o fim do período e o repasse:")
print(ifood["dias_ate_repasse"].value_counts())

# d) duplicados e semanas faltando (cada período começa no dia seguinte ao fim do anterior?)
ifood = ifood.sort_values("periodo_inicio").reset_index(drop=True)
print("\nPeríodos duplicados:", ifood.duplicated(subset=["periodo_inicio"]).sum())
salto = (ifood["periodo_inicio"] - ifood["periodo_fim"].shift(1)).dt.days
print("Semanas com buraco ou sobreposição:")
display(ifood.loc[salto.notna() & (salto != 1), ["periodo_inicio", "periodo_fim", "data_repasse"]])

display(ifood.head())

Falhas (vazios depois da conversão):
data_repasse      0
qtd_pedidos       0
valor_vendas      0
comissao_ifood    0
taxa_pagamento    0
valor_repasse     0
periodo_inicio    0
periodo_fim       0
dtype: int64

Linhas em que a conta não fecha: 0

Dias por período:
dias_periodo
7    74
Name: count, dtype: int64

Dias entre o fim do período e o repasse:
dias_ate_repasse
8    74
Name: count, dtype: int64

Períodos duplicados: 0
Semanas com buraco ou sobreposição:


,periodo_inicio,periodo_fim,data_repasse


,data_repasse,qtd_pedidos,valor_vendas,comissao_ifood,taxa_pagamento,valor_repasse,periodo_inicio,periodo_fim,dif_conta,dias_periodo,dias_ate_repasse
0,2025-06-04,16,949.08,182.76,37.43,728.89,2025-05-21,2025-05-27,0.0,7,8
1,2025-06-11,17,960.41,184.93,37.88,737.60,2025-05-28,2025-06-03,0.0,7,8
2,2025-06-18,15,860.91,165.78,33.95,661.18,2025-06-04,2025-06-10,0.0,7,8
3,2025-06-25,14,824.84,158.83,32.53,633.48,2025-06-11,2025-06-17,0.0,7,8
4,2025-07-02,16,913.40,175.89,36.02,701.49,2025-06-18,2025-06-24,0.0,7,8


In [32]:
#organizando o df
primeiras = ["periodo_inicio", "periodo_fim", "data_repasse"]
resto = [c for c in ifood.columns if c not in primeiras]

ifood = ifood[primeiras + resto]
ifood.columns.tolist()
ifood.head()

,periodo_inicio,periodo_fim,data_repasse,qtd_pedidos,valor_vendas,comissao_ifood,taxa_pagamento,valor_repasse,dif_conta,dias_periodo,dias_ate_repasse
0,2025-05-21,2025-05-27,2025-06-04,16,949.08,182.76,37.43,728.89,0.0,7,8
1,2025-05-28,2025-06-03,2025-06-11,17,960.41,184.93,37.88,737.60,0.0,7,8
2,2025-06-04,2025-06-10,2025-06-18,15,860.91,165.78,33.95,661.18,0.0,7,8
3,2025-06-11,2025-06-17,2025-06-25,14,824.84,158.83,32.53,633.48,0.0,7,8
4,2025-06-18,2025-06-24,2025-07-02,16,913.40,175.89,36.02,701.49,0.0,7,8


In [33]:
#CONFERENCIA DOS VALORESDO IFOOD COM OO EXTRATO DO BANCO
#Esses valores tem uqe entrar no banco na data de repasse

# extrato tratado (
ext =pd.read_csv(r'C:\Users\nadia\OneDrive\Desktop\Leticia\Documentos\Projetos\cliente_confeitaria\Analise_financeira_confeitaria\Dados\tratados\extratos_tratados.csv', sep=";", decimal=",",
                  encoding="utf-8-sig", parse_dates=["Data"], dayfirst=True)

# entradas do iFood no banco
ext_ifood = ext[ext["Descrição"].str.contains("IFOOD", case=False, na=False)]
print("Lançamentos do iFood no extrato:", len(ext_ifood))
print("Repasses no relatório:          ", len(ifood))
print(ext_ifood["Descrição"].unique())   # confira se só pegou o que é repasse

# cruzamento por data + valor
comp = ifood.merge(
    ext_ifood[["Data", "Valor", "Descrição"]],
    left_on=["data_repasse", "valor_repasse"],
    right_on=["Data", "Valor"],
    how="outer",
    indicator=True,
)
print(comp["_merge"].value_counts())
#todas as datas batem com os pagamentos. 

Lançamentos do iFood no extrato: 74
Repasses no relatório:           74
<StringArray>
['TED RECEBIDA - IFOOD.COM AGENCIA DE RESTAURANTES']
Length: 1, dtype: str
_merge
both          74
left_only      0
right_only     0
Name: count, dtype: int64


In [35]:
ifood_final = ifood.drop(columns=["dif_conta", "dias_periodo", "dias_ate_repasse"])

ifood_final.to_csv(
    path + r"\tratados\ifood_tratado.csv",
    sep=";", decimal=",", date_format="%d/%m/%Y",
    encoding="utf-8-sig", index=False,
)
print(ifood_final.shape)

(74, 8)
